In [ ]:
import scanpy as sc
import scvi
import seaborn as sns
import numpy as np
import pandas as pd

## Counting

In [ ]:
myeloid = sc.read_h5ad("myeloid_integrated_final_label.h5ad")
myeloid.obs.Sample.unique().tolist()

In [ ]:
def map_condition(x):
    if 'Tumor' in x:
        return 'Tumor'
    elif 'Involved' in x:
        return 'Involved'
    elif 'Distal' in x:
        return 'Distal'
    else:
        return 'Benign'

myeloid.obs['condition'] = myeloid.obs.Sample.map(map_condition)
myeloid.obs

In [ ]:
num_tot_cells = myeloid.obs.groupby(['Sample']).count()
num_tot_cells = dict(zip(num_tot_cells.index, num_tot_cells.doublet))
num_tot_cells

In [ ]:
cell_type_counts = myeloid.obs.groupby(['Sample', 'condition', 'scanvi_labels_annotation_model']).count()
cell_type_counts = cell_type_counts[cell_type_counts.sum(axis = 1) > 0].reset_index()
cell_type_counts = cell_type_counts[cell_type_counts.columns[0:4]]
cell_type_counts.columns

In [ ]:
cell_type_counts['total_cells'] = cell_type_counts.Sample.map(num_tot_cells).astype(int)

cell_type_counts['frequency'] = cell_type_counts.doublet / cell_type_counts.total_cells

cell_type_counts

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize = (10,4))

ax = sns.boxplot(data = cell_type_counts, x = 'scanvi_labels_annotation_model', y = 'frequency', hue = 'condition')

plt.xticks(rotation = 35, rotation_mode = 'anchor', ha = 'right')

plt.show()


## Which genesets are contributing most to myeloid cells of each condition?

In [ ]:
sc.pl.umap(myeloid, color='leiden')

In [ ]:
myeloid


In [ ]:
sc.pl.umap(myeloid, color=['scanvi_labels_annotation_model', 'scanvi_labels_annotation_model_refined'])

In [ ]:
sc.pl.umap(myeloid, color='Sample')

In [ ]:
sc.pl.umap(myeloid, color='condition')  # tissue = distal, involved, tumor
sc.tl.rank_genes_groups(myeloid, groupby='condition', use_raw=True)
sc.pl.rank_genes_groups(myeloid)


## GSVA

In [ ]:
gp.get_library_name()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import gseapy as gp
from scipy.stats import zscore
import seaborn as sns
import matplotlib.pyplot as plt

library_list = ['CellMarker_2024']
# Ensure consistent casing

# Step 1: Define conditions and pseudobulk expression
# ordered_conditions = ['Benign', 'Distal', 'Involved', 'Tumor']
# condition_key = 'condition'  # Replace with your actual adata.obs column for condition

ordered_conditions = myeloid.obs['condition'].cat.categories.tolist()
condition_key = 'condition'  # Replace with your actual adata.obs column for condition


# Step 2: Pseudobulk average expression per condition
pseudobulk_expr = {}

for cond in ordered_conditions:
    cells = adata.obs[adata.obs[condition_key] == cond].index
    expr_subset = adata[cells].X.toarray() if hasattr(adata[cells].X, "toarray") else adata[cells].X
    expr_df = pd.DataFrame(expr_subset.T, index=adata.var_names)
    pseudobulk_expr[cond] = expr_df.mean(axis=1)

# Step 3: Build combined expression matrix
expr_matrix = pd.DataFrame(pseudobulk_expr)

# Step 4: Run GSVA
gsva_result = gp.gsva(
    data=expr_matrix,
    gene_sets=library_list,
    method='gsva',
    outdir=None,
    no_plot=True,
    verbose=True
)

# Step 5: Extract GSVA scores matrix
gsva_scores = gsva_result.res2d.pivot(index='Term', columns='Name', values='ES')

# Step 6: Ensure correct column order
gsva_scores = gsva_scores[[col for col in ordered_conditions if col in gsva_scores.columns]]

# Step 7: Filter top pathways by variance
top_var_pathways = gsva_scores.var(axis=1).sort_values(ascending=False).head(10).index

gsva_var_filtered = gsva_scores.loc[top_var_pathways]

# Step 8: Z-score normalization across conditions
# Ensure all values are floats
gsva_var_filtered = gsva_var_filtered.astype(float)

# Drop rows with all NaNs just in case
gsva_var_filtered = gsva_var_filtered.dropna(how='all')

# Apply z-score across rows (i.e., across conditions)
from scipy.stats import zscore
gsva_z = gsva_var_filtered.apply(lambda x: zscore(x, nan_policy='omit'), axis=1)


# Format annotations
annotations = gsva_z.round(2).astype(str)

# Set modern seaborn style
sns.set_theme(style="white")

# Create heatmap
plt.figure(figsize=(10, 12))
ax = sns.heatmap(
    gsva_z,
    cmap="vlag",  # Modern diverging colormap
    annot=annotations,
    fmt="",
    annot_kws={"size": 8, "color": "black"},
    linewidths=0.5,
    linecolor="lightgray",
    cbar_kws={"label": "Z-score"},
    xticklabels=True,
    yticklabels=True,
)

# Aesthetic tweaks
ax.set_xticklabels(ax.get_xticklabels(), fontsize=10, rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('')
sns.despine(left=True, bottom=True)

plt.title("GSVA Z-scores of Top Pathways", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

sc.pl.umap(myeloid,
           color=['leiden'],
           legend_loc='on data',
           legend_fontsize=12,
           legend_fontoutline=2)
sc.pl.umap(myeloid, color=['Sample', 'condition'])

In [ ]:
for library in library_list:
    # Step 2: Download the gene set library
    ref = gp.get_library(name=library)  # or 'Reactome_Pathways_2024'

    for pathway in top_var_pathways:
        pathway = pathway.split("__")[1]
        pathway_genes = ref[pathway]

        # Ensure gene names are in your dataset
        genes_present = [gene for gene in pathway_genes if gene in myeloid.var_names]
        print(f"Genes found in dataset: {genes_present}")

        # Score cells based on this pathway
        sc.tl.score_genes(myeloid, gene_list=genes_present, score_name=f'{pathway}_pathway_score', use_raw=True)

        # UMAP visualization
        sc.pl.umap(myeloid, color=[f'{pathway}_pathway_score','condition', 'leiden'], cmap='viridis', title=f'{pathway} Pathway Score')


In [ ]:
myeloid

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import gseapy as gp
from scipy.stats import zscore
import seaborn as sns
import matplotlib.pyplot as plt

library_list = ['ChEA_2022']
# Ensure consistent casing

# Step 1: Define conditions and pseudobulk expression
# ordered_conditions = ['Benign', 'Distal', 'Involved', 'Tumor']
# condition_key = 'condition'  # Replace with your actual adata.obs column for condition

ordered_conditions = myeloid.obs['scanvi_labels_annotation_model_refined'].cat.categories.tolist()
condition_key = 'scanvi_labels_annotation_model_refined'  # Replace with your actual adata.obs column for condition


# Step 2: Pseudobulk average expression per condition
pseudobulk_expr = {}

for cond in ordered_conditions:
    cells = adata.obs[adata.obs[condition_key] == cond].index
    expr_subset = adata[cells].X.toarray() if hasattr(adata[cells].X, "toarray") else adata[cells].X
    expr_df = pd.DataFrame(expr_subset.T, index=adata.var_names)
    pseudobulk_expr[cond] = expr_df.mean(axis=1)

# Step 3: Build combined expression matrix
expr_matrix = pd.DataFrame(pseudobulk_expr)

# Step 4: Run GSVA
gsva_result = gp.gsva(
    data=expr_matrix,
    gene_sets=library_list,
    method='gsva',
    outdir=None,
    no_plot=True,
    verbose=True
)

# Step 5: Extract GSVA scores matrix
gsva_scores = gsva_result.res2d.pivot(index='Term', columns='Name', values='ES')

# Step 6: Ensure correct column order
gsva_scores = gsva_scores[[col for col in ordered_conditions if col in gsva_scores.columns]]

# Step 7: Filter top pathways by variance
top_var_pathways = gsva_scores.var(axis=1).sort_values(ascending=False).head(10).index

gsva_var_filtered = gsva_scores.loc[top_var_pathways]

# Step 8: Z-score normalization across conditions
# Ensure all values are floats
gsva_var_filtered = gsva_var_filtered.astype(float)

# Drop rows with all NaNs just in case
gsva_var_filtered = gsva_var_filtered.dropna(how='all')

# Apply z-score across rows (i.e., across conditions)
from scipy.stats import zscore
gsva_z = gsva_var_filtered.apply(lambda x: zscore(x, nan_policy='omit'), axis=1)


# Format annotations
annotations = gsva_z.round(2).astype(str)

# Set modern seaborn style
sns.set_theme(style="white")

# Create heatmap
plt.figure(figsize=(10, 12))
ax = sns.heatmap(
    gsva_z,
    cmap="vlag",  # Modern diverging colormap
    annot=annotations,
    fmt="",
    annot_kws={"size": 8, "color": "black"},
    linewidths=0.5,
    linecolor="lightgray",
    cbar_kws={"label": "Z-score"},
    xticklabels=True,
    yticklabels=True,
)

# Aesthetic tweaks
ax.set_xticklabels(ax.get_xticklabels(), fontsize=10, rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('')
sns.despine(left=True, bottom=True)

plt.title("GSVA Z-scores of Top Pathways", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

sc.pl.umap(myeloid,
           color=['leiden'],
           legend_loc='on data',
           legend_fontsize=12,
           legend_fontoutline=2)
sc.pl.umap(myeloid, color=['Sample', 'condition'])

In [ ]:
for library in library_list:
    # Step 2: Download the gene set library
    ref = gp.get_library(name=library)  # or 'Reactome_Pathways_2024'

    for pathway in top_var_pathways:
        pathway = pathway.split("__")[1]
        pathway_genes = ref[pathway]

        # Ensure gene names are in your dataset
        genes_present = [gene for gene in pathway_genes if gene in myeloid.var_names]
        print(f"Genes found in dataset: {genes_present}")

        # Score cells based on this pathway
        sc.tl.score_genes(myeloid, gene_list=genes_present, score_name=f'{pathway}_pathway_score', use_raw=True)

        # UMAP visualization
        sc.pl.umap(myeloid, color=[f'{pathway}_pathway_score','condition', 'leiden'], cmap='viridis', title=f'{pathway} Pathway Score')


## M2, CD36, and Metabolism

In [ ]:
# Define M2 macrophage genes (literature-based)
m2_genes = ['CD163', 'MRC1', 'IL10', 'ARG2', 'MSR1']  # Add CD36 for convenience

# Fatty acid metabolism genes (example subset from Hallmark FA metabolism)
fatty_acid_genes = [
    'CD36', 'CPT1A', 'ACSL1', 'SLC27A1', 'FABP4', 'PPARG', 'LPL',
    'ACADVL', 'EHHADH', 'ACADM', 'HADH', 'ACOX1'
]


In [ ]:
import scanpy as sc

# M2 signature score
sc.tl.score_genes(myeloid, gene_list=m2_genes, score_name='M2_score', use_raw=False)

# Fatty acid metabolism signature
sc.tl.score_genes(myeloid, gene_list=fatty_acid_genes, score_name='FA_score', use_raw=False)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a dataframe
df = myeloid.obs[['condition', 'M2_score', 'FA_score']].copy()
df['CD36'] = myeloid[:, 'CD36'].X.toarray().flatten() if hasattr(myeloid[:, 'CD36'].X, "toarray") else myeloid[:, 'CD36'].X.A1

# Boxplot of M2 score
plt.figure(figsize=(8, 5))
sns.boxplot(x='condition', y='M2_score', data=df, palette='Set2')
plt.title('M2 Macrophage Score by Condition')
plt.ylabel('M2 Score')
plt.xlabel('')
plt.show()

# Boxplot of CD36 expression
plt.figure(figsize=(8, 5))
sns.boxplot(x='condition', y='CD36', data=df, palette='Set3')
plt.title('CD36 Expression by Condition')
plt.ylabel('CD36 Expression')
plt.xlabel('')
plt.show()

# Scatterplot: M2 score vs CD36
sns.lmplot(data=df, x='M2_score', y='CD36', hue='condition', aspect=1.3)
plt.title('CD36 vs M2 Signature Score')
plt.show()


In [ ]:
import scipy.stats as stats

# Compare M2 score in Tumor vs other conditions
tumor = df[df['condition'] == 'Tumor']['M2_score']
other = df[df['condition'] != 'Tumor']['M2_score']
stat, p = stats.ttest_ind(tumor, other)
print(f"Tumor vs others M2 score t-test: p = {p:.4f}")


In [ ]:
import scipy.stats as stats

# Compare M2 score in Tumor vs other conditions
tumor = df[df['condition'] == 'Tumor']['M2_score']
other = df[df['condition'] != 'Tumor']['M2_score']
stat, p = stats.ttest_ind(tumor, other)
print(f"Tumor vs others M2 score t-test: p = {p:.4f}")


In [ ]:
sc.pl.umap(myeloid, color=['M2_score', 'CD36', 'FA_score', 'condition'], cmap='viridis', vmin=0)


## Genes alignining with M2 Polarization

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

# Step 1: Get M2 scores and gene expression matrix
m2_scores = myeloid.obs['M2_score'].values
gene_expr = myeloid.X.toarray() if hasattr(myeloid.X, "toarray") else myeloid.X
gene_names = myeloid.var_names

# Step 2: Correlate each gene with M2 score
correlation_results = []

for i, gene in enumerate(gene_names):
    expr_values = gene_expr[:, i]
    rho, pval = spearmanr(expr_values, m2_scores)
    correlation_results.append((gene, rho, pval))

# Step 3: Convert to DataFrame and sort
cor_df = pd.DataFrame(correlation_results, columns=['gene', 'spearman_rho', 'pval'])
cor_df = cor_df.dropna().sort_values(by='spearman_rho', ascending=False)

# Top positively and negatively correlated genes
top_pos = cor_df.head(20)
top_neg = cor_df.tail(20)

print("Top 10 positively correlated genes with M2:")
print(top_pos.head(10))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare DataFrame
df = myeloid.obs[['M2_score']].copy()
df['CD36'] = myeloid[:, 'CD36'].X.toarray().flatten() if hasattr(myeloid[:, 'CD36'].X, "toarray") else myeloid[:, 'CD36'].X.A1

# Scatterplot with regression
plt.figure(figsize=(7, 5))
sns.regplot(x='M2_score', y='CD36', data=df, scatter_kws={'alpha': 0.3, 's': 10}, line_kws={'color': 'red'})
plt.title('CD36 Expression vs. M2 Signature Score')
plt.xlabel('M2 Score')
plt.ylabel('CD36 Expression')
plt.tight_layout()
plt.show()


In [ ]:
# Add condition if not already there
df['condition'] = myeloid.obs['condition']

# Violin plot: M2
plt.figure(figsize=(8, 4))
sns.violinplot(data=df, x='condition', y='M2_score', inner='box', palette='Set2')
plt.title('M2 Score by Condition')
plt.ylabel('M2 Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Violin plot: CD36
plt.figure(figsize=(8, 4))
sns.violinplot(data=df, x='condition', y='CD36', inner='box', palette='Set3')
plt.title('CD36 Expression by Condition')
plt.ylabel('CD36 Expression')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
sc.pl.umap(myeloid, color=['CTSS', 'CX3CR1', 'NFKBIA', 'CXCL3'], cmap='viridis', vmax='p99', size=25)


In [ ]:
sns.jointplot(data=df, x='M2_score', y='CD36', kind='kde', fill=True, cmap='Blues')
plt.suptitle('Density of CD36 vs M2 Signature Score', y=1.02)
plt.show()


In [ ]:
df['high_M2'] = df['M2_score'] > df['M2_score'].quantile(0.75)
df['low_CD36'] = df['CD36'] < df['CD36'].quantile(0.25)
df['group'] = df.apply(lambda x: f"{'M2-hi' if x.high_M2 else 'M2-lo'}|{'CD36-lo' if x.low_CD36 else 'CD36-hi'}", axis=1)

sns.countplot(data=df[df['high_M2']], x='group', hue='condition')
plt.title('Composition of High-M2 Cells by CD36 Status and Condition')


In [ ]:
# bcr cluster 2 score

c2_genes = ['NDUFAF6', 'RAD21', 'DCAF13', 'COPS5', 'SCRIB', 'NUDCD1', 'SLC25A32', 'YWHAZ']

sc.tl.score_genes(adata, gene_list=c2_genes, score_name='C2_score', use_raw=False)


sc.pl.umap(adata, color=['C2_score', 'condition', 'scanvi_labels_annotation_model'], legend_loc='on data', cmap='viridis', vmin=0)


## Trajectory Inference

In [ ]:
from moscot.problems.time import TemporalProblem

import cellrank as cr
import scanpy as sc
from cellrank.kernels import RealTimeKernel

import matplotlib.pyplot as plt
import seaborn as sns

# Set Scanpy-style figure defaults manually
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'
sns.set_style("white")

cr.settings.verbosity = 2

print(cr.__version__)

In [ ]:
# myeloid = sc.read_h5ad("myeloid_integrated_final_label.h5ad")
myeloid = sc.read_h5ad("myeloid_FINAL.h5ad")
adata = myeloid

In [ ]:
# # Step 1: Keep full gene set
# adata.raw = adata  # raw holds full 20k genes

# # Step 2: Identify HVGs
# sc.pp.highly_variable_genes(adata, subset=True, inplace=True)

# # Step 3: PCA and neighbors ONLY on HVGs
# sc.pp.normalize_total(adata)
# sc.pp.log1p(adata)
# sc.pp.scale(adata, max_value=10)
# sc.pp.pca(adata, use_highly_variable=True)
# sc.pp.neighbors(adata, n_neighbors=50, n_pcs=50)
# sc.tl.leiden(adata, resolution=0.5)
# sc.tl.umap(adata)

In [ ]:
sc.pl.embedding(
    adata, basis="X_umap", color=["condition", "leiden", "scanvi_labels_annotation_model", "Sample"]
)

In [ ]:
# adata.write_h5ad('myeloid_FINAL.h5ad')

In [ ]:
adata

### Export h5ad for celloracle ###

In [ ]:
tp = TemporalProblem(adata)

In [ ]:
tp = tp.score_genes_for_marginals(
    gene_set_proliferation="human", gene_set_apoptosis="human"
)

In [ ]:
sc.pl.embedding(
    adata, basis="X_umap", color=["condition", "proliferation", "apoptosis"]
)

In [ ]:
# Create a dictionary mapping your categorical conditions to numbers
condition_map = {
    'Benign': 1,
    'Distal': 2,
    'Involved': 3,
    'Tumor': 4
}

# Apply the mapping
adata.obs['condition_numeric'] = adata.obs['condition'].map(condition_map).astype(float)

sc.pl.embedding(
    adata, basis="X_umap", color=["condition_numeric"]
)

In [ ]:
tp = tp.prepare(time_key="condition_numeric")

In [ ]:
tp = tp.solve(epsilon=1e-3, tau_a=0.95, initializer='rank2', rank=30, scale_cost='mean')

In [ ]:
tmk = RealTimeKernel.from_moscot(tp)

In [ ]:
tmk.compute_transition_matrix(self_transitions="all", conn_weight=0.2, threshold="auto")

In [ ]:
tmk.plot_random_walks(
    max_iter=100,
    start_ixs={"condition_numeric": 1},
    basis="X_umap",
    seed=0,
    dpi=150,
    size=30,
)

## Drivers

In [ ]:
g = cr.estimators.GPCCA(tmk)
print(tmk)

In [ ]:
g.fit(cluster_key="condition_numeric", n_states=4)

In [ ]:
import matplotlib.pyplot as plt

# Your list of terminal state names
terminal_states = ['4.0_1', '4.0_2', '4.0_3', '4.0_4']

# Define a list of colors (same length as terminal_states)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Use your own hex codes or color names

# Assign the colors to the appropriate .uns key
adata.uns['terminal_states_colors'] = colors

# Set the terminal states again (this should use your new colors)
g.set_terminal_states(states=terminal_states)

# Plot with updated colors
g.plot_macrostates(which="terminal", legend_loc="right", size=100)

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot=False)

In [ ]:
g.plot_fate_probabilities(same_plot=True)

In [ ]:
cr.pl.circular_projection(adata, keys=["condition_numeric", "leiden"], legend_loc="right")

In [ ]:
sc.pl.umap(adata, color=['leiden', 'scanvi_labels_annotation_model_refined', 'condition', 'Sample'], cmap='viridis', vmin=0)

In [ ]:
# leiden_states = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "10"]
leiden_states = ["Benign", "Distal", "Involved", "Tumor"]
sc.pl.embedding(
    adata, basis="umap", color="condition", groups=leiden_states, legend_loc="right"
)

In [ ]:
cr.pl.aggregate_fate_probabilities(
    adata,
    mode="violin",
    lineages=["4.0_1"],
    cluster_key="condition",
    clusters=leiden_states,
)

In [ ]:
driver_clusters = ["Distal"]
# leiden_states = ["3"]

tam_df = g.compute_lineage_drivers(
    lineages=["4.0_4"], cluster_key="condition", clusters=leiden_states
)
# Automatically find the q-value column
qval_col = [col for col in tam_df.columns if col.endswith("_qval")][0]

# Filter for genes with q-value < 0.05
filtered_tam_df = tam_df[tam_df[qval_col] < 0.05]

# # Display all rows
# pd.set_option("display.max_rows", None)
# filtered_tam_df.shape
# filtered_tam_df

In [ ]:
# adata.obs["fate_probabilities_TAM"] = g.fate_probabilities["4.0_4"].X.flatten()

# sc.pl.embedding(
#     adata,
#     basis="umap",
#     color=["fate_probabilities_TAM"] + list(tam_df.index[:8]),
#     color_map="viridis",
#     s=50,
#     ncols=3,
#     vmax="p96",
#     use_raw=True
# )

In [ ]:
# import numpy as np
# import pandas as pd

# lineage_x = "4.0_2"
# lineage_y = "4.0_4"
# top_n = 20
# driver_clusters = ["Distal"]


# # Step 1: Compute drivers
# driver_df = g.compute_lineage_drivers(
#     lineages=[lineage_x, lineage_y],
#     cluster_key="condition",
#     clusters=driver,
# )

# # Step 2: Get top N genes with highest absolute correlation
# top_x = (
#     driver_df[f"{lineage_x}_corr"]
#     .abs()
#     .sort_values(ascending=False)
#     .head(top_n)
#     .index
#     .tolist()
# )
# top_y = (
#     driver_df[f"{lineage_y}_corr"]
#     .abs()
#     .sort_values(ascending=False)
#     .head(top_n)
#     .index
#     .tolist()
# )

# # Optional: Remove overlapping genes
# top_x = [g for g in top_x if g not in top_y]
# top_y = [g for g in top_y if g not in top_x]

# # Step 3: Create gene sets to label
# genes_oi = {
#     f"Top_{lineage_x}": top_x,
#     f"Top_{lineage_y}": top_y,
# }

# # Compute mean expression using full gene set from raw
# mean_expr = np.asarray(adata.raw.X.mean(axis=0)).flatten()

# # Map the raw var index to the current filtered var
# mean_expr_series = pd.Series(mean_expr, index=adata.raw.var_names)
# mean_expr_subset = mean_expr_series[adata.var_names]  # Align indices

# # Store in adata.var (not .raw.var)
# adata.var["mean expression"] = mean_expr_subset.values



# # Step 5: Visualize
# g.plot_lineage_drivers_correlation(
#     lineage_x=lineage_x,
#     lineage_y=lineage_y,
#     adjust_text=True,
#     gene_sets=genes_oi,
#     color=f"{driver} to terminal mean expression",
#     legend_loc="none",
#     figsize=(5, 5),
#     dpi=150,
#     fontsize=9,
#     size=50,
# )


In [ ]:
import pandas as pd
import os

# Define clusters and lineages
driver_clusters = ["Benign", "Distal", "Involved", "Tumor"]
terminal_lineages = ["4.0_1", "4.0_2", "4.0_3", "4.0_4"]
top_n = 20

# Output folder
output_dir = "driver_gene_tables"
os.makedirs(output_dir, exist_ok=True)

for driver in driver_clusters:
    for lineage in terminal_lineages:
        # Step 1: Compute drivers for one terminal lineage
        driver_df = g.compute_lineage_drivers(
            lineages=[lineage],
            cluster_key="condition",
            clusters=driver,
        )

        # Step 2: Filter and sort by correlation
        sort_col = f"{lineage}_corr"
        output_cols = [
            f"{lineage}_corr",
            f"{lineage}_pval",
            f"{lineage}_qval",
            f"{lineage}_ci_low",
            f"{lineage}_ci_high"
        ]
        top_df = (
            driver_df[output_cols]
            .sort_values(by=sort_col, ascending=False)
        )

        # Step 3: Export to CSV
        filename = f"{driver}_to_{lineage}_drivers.csv"
        top_df.to_csv(os.path.join(output_dir, filename))


## Gene Set Analysis

In [ ]:
import magic
import scanpy as sc

# Step 1: Assume adata is already PCA'd and has neighbors computed
# If not: run sc.pp.neighbors(adata), etc.

# Step 2: Run MAGIC
magic_operator = magic.MAGIC()
magic_result = magic_operator.fit_transform(adata)  # this is an AnnData object

# Step 3: Save the imputed matrix (not the full AnnData) into .layers
adata.layers["magic"] = magic_result.X  # ✅ the actual expression matrix

# Optional: update var_names if needed
adata.var_names = magic_result.var_names.copy()


In [ ]:
print(type(adata.layers["magic"]))       # should be <class 'numpy.ndarray'> or similar
print(adata.layers["magic"].shape)       # should match (n_cells, n_genes)
print(adata.layers["magic"].dtype)       # should be float32 or float64

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import palantir

# Step 1: Get diffusion map embedding
dm_res = adata.obsm["X_umap"]
dm_df = pd.DataFrame(dm_res, index=adata.obs_names)

# Step 2: Subset to Benign cells
benign_cells = adata.obs[adata.obs['condition'] == "Benign"].index
benign_dm = dm_df.loc[benign_cells]

# Step 3: Compute centroid of Benign cells in diffusion space
centroid = benign_dm.mean(axis=0).values.reshape(1, -1)

# Step 4: Find cell closest to the centroid
all_dm = dm_df.values
distances = cdist(all_dm, centroid)
closest_idx = np.argmin(distances)
start_cell = dm_df.index[closest_idx]

print(f"Selected root cell (Benign centroid): {start_cell}")

# Step 5: Run Palantir with that root
pr_res = palantir.core.run_palantir(dm_df, start_cell)
adata.obs["palantir_pseudotime"] = pr_res.pseudotime.loc[adata.obs_names]


In [ ]:
sc.pl.embedding(
    adata,
    basis="umap",
    color=["condition", "palantir_pseudotime", "leiden"],
    color_map="gnuplot",
    legend_loc="on data",
)

In [ ]:
sc.pl.violin(adata, keys=["palantir_pseudotime"], groupby="condition", rotation=90)

In [ ]:
model = cr.models.GAMR(adata, n_knots=6, smoothing_penalty=10.0)

In [ ]:
cr.pl.gene_trends(
    adata,
    model=model,
    data_key="magic",
    genes=["KLF6"],
    same_plot=True,
    ncols=2,
    time_key="palantir_pseudotime",
    hide_cells=True,
    weight_threshold=(1e-3, 1e-3),
)


In [ ]:
sc.pl.embedding(
    adata,
    basis="umap",
    color=["KLF6", "leiden", "condition"],
    color_map="gnuplot",
    legend_loc="on data",
)

In [ ]:
# compute putative drivers for the Beta trajectory
beta_drivers = g.compute_lineage_drivers(lineages="4.0_4")

# plot heatmap
cr.pl.heatmap(
    adata,
    model=model,  # use the model from before
    lineages="4.0_4",
    cluster_key="condition_numeric",
    show_fate_probabilities=True,
    data_key="magic",
    genes=beta_drivers.head(50).index,
    time_key="palantir_pseudotime",
    figsize=(12, 10),
    show_all_genes=True,
    weight_threshold=(1e-3, 1e-3),
)

In [ ]:
pd.set_option('display.max_rows', None)

display(beta_drivers[beta_drivers["4.0_2_qval"] < 0.05])

pd.reset_option('display.max_rows')
